# Downloading GUNW products using ariaDownload.py

**Author**: Brett A. Buzzanga, David Bekaert - Jet Propulsion Laboratory

This notebook demonstrates how to use the productAPI.py command line tool to download Sentinel 1 ARIA Geocoded UNWrapped interferogram (**GUNW**) products.  A detailed overview of the ARIA GUNW product with respect to processing, formatting, sampling, and data layers can be found on the [ARIA website](https://aria.jpl.nasa.gov/node/97).

The **`ariaDownload.py`** program wraps around the NASA's ASF DAAC API and [Bulk Download Service](https://bulk-download.asf.alaska.edu/help). The ASF Bulk Download Service handles most of the heavy lifting of the data-download and will conveniently skip previously downloaded files, and re-download partially downloaded files.  
In this notebook, we will demonstrate **`ariaDownload.py`** functionality along track 4, which intersects the U.S. East Coast in southeastern Virginia.


<div class="alert alert-warning">
<b>Potential download failure:</b> 
GUNW products are hosted at the NASA ASF DAAC. Downloading them requires a NASA Earthdata URS user login and requires users to add "GRFN Door (PROD)" to their URS approved applications

<b>Login Credentials:</b>
Save your user-name and password to a `.netrc` file in your `$HOME` directory, or pass the combination explicitly using `ariaDownload.py --user <user> --pass <pass>`.


To create a .netrc file, pass your earthdata credentials by running the cell below
</div>

In [1]:
import os

# create .netrc if it does not exist    
if not os.path.exists(os.path.expanduser('~/.netrc')):
    print('NEEDED To Download ARIA GUNWs: \n Link to create account : https://urs.earthdata.nasa.gov/')
    earthdata_user = input('Please type your Earthdata username:')
    earthdata_user = str(earthdata_user)
    earthdata_password = input('Please type your Earthdata password:')
    earthdata_password = str(earthdata_password)
    os.system('echo machine urs.earthdata.nasa.gov login "{usern}" password "{passwd}" > ~/.netrc; chmod 600 ~/.netrc'.format( \
              usern = earthdata_user, passwd = earthdata_password))

In [2]:
## Defining the home and data directories at the processing location
home_dir = os.getcwd()
tutorial_home_dir = os.path.abspath(os.path.join(home_dir, ""))
tools_dir = os.path.join(tutorial_home_dir,'tools',"bin")
print("home directory: ", tutorial_home_dir)
print("tools directory: ", tools_dir)

home directory:  C:\Users\nblin\OneDrive - University of Massachusetts\Postdoc\04-Python codes\ARIA-tools
tools directory:  C:\Users\nblin\OneDrive - University of Massachusetts\Postdoc\04-Python codes\ARIA-tools\tools\bin


In [12]:
os.getcwd()

'C:\\Users\\nblin\\OneDrive - University of Massachusetts\\Postdoc\\04-Python codes\\ARIA-tools\\tools\\bin'

In [3]:
os.chdir(tools_dir)

## Overview of the ariaDownload.py program

Running **`ariaDownload.py`** with no options, or with **`-h`**, will show the parameters options as well as some examples. At minimum, users need to specify a spatial constraint either as a track number or bounding box (can be a shapefile).

Let us explore what some of the other options are:

In [14]:
!python ariaDownload.py --help

usage: ariaDownload.py [-h] [-o {Download,Count,Url}] [-t TRACK] [-b BBOX]
                       [-w WD] [-s START] [-e END] [-u USER] [-p PASSW]
                       [--mission {S1,NISAR}] [-l DAYSLT] [-m DAYSGT]
                       [-nt NUM_THREADS] [-i IFG] [-d FLIGHTDIR]
                       [--version VERSION] [-v] [--log-level LOG_LEVEL]

Command line interface to download Sentinel-1/NISAR GUNW products from the ASF DAAC. 

options:
  -h, --help            show this help message and exit
  -o, --output {Download,Count,Url}
                        Output type. Default="Download". Use "Url" for
                        ingestion to aria*.py
  -t, --track TRACK     track to download; single number (including leading
                        zeros) or comma separated
  -b, --bbox BBOX       Lat/Lon Bounding SNWE, or GDAL-readable file
                        containing POLYGON geometry.
  -w, --workdir WD      Specify directory to deposit all outputs. Default is
               

### Count the number of products

To get a count of the number of products, without downloading data, provide the **`--output`** option with the **`count`** argument. To get information on the exact product filenames also include the verbose option **`-v`**.

In [13]:
!python ariaDownload.py --track 156 --output count

C:\Users\nblin\OneDrive - University of Massachusetts\Postdoc\04-Python codes\ARIA-tools\tools\bin\ariaDownload.py:280: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  return asf_search.geo_search(
2025-06-25 09:02:00,181 - ariaDownload.py - INFO - Found -- 7897 -- products


In [ ]:
!python ariaDownload.py --track 156 --bbox "-24.5 -22.0 -69.0 -67.5" \
                     --start 20150101 --end 20200101 --daysmore 30 --version '3_0_1'

### Generate list of virtual products from ASF S3 bucket (BETA)

To generate a textfile containing a list of product URLs from the ASF S3 bucket, without downloading data, provide the **`--output`** option with the **`url`** argument. To get information on the exact product filenames also include the verbose option **`-v`**. Extracting layers virtually by leveraging this list of URLs is currently only supported by systems with the following packages: Linux kernel >=4.3 and libnetcdf >=4.5 

In [ ]:
!ariaDownload.py --track 4 --output url

We can now have a look at the generated textfile, which contains the URLs of all standard products over the specified track. As there are a lot of products we will only have a look at the first 10.

In [ ]:
!head -n 10 products/track4_0.txt

In [ ]:
current_dir = os.getcwd()

aria_dir = os.path.join(current_dir, "tools","bin")
print(aria_dir)
os.makedirs(aria_dir, exist_ok=True)